# Benchamark utilizando AutoML. En este caso con flaml.

In [1]:
# Cargar librerías necesarias

!pip install -U "flaml[automl]"
import numpy as np
import pandas as pd
if not hasattr(np, "NaN"): np.NaN = np.nan  # hotfix NumPy 2.x
from flaml import AutoML
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    average_precision_score, f1_score, precision_score, recall_score,
    roc_auc_score, accuracy_score
)



In [2]:
uploaded = files.upload()

Saving df_trans.pkl to df_trans (2).pkl


In [3]:
df = pd.read_pickle("df_trans.pkl")

In [4]:
# Verificar el dataframe
print(df.head())
print(df.shape)

   Diabetes_binary  HighBP  HighChol  CholCheck       BMI  Smoker  Stroke  \
0              0.0     1.0       0.0        1.0 -0.553424     0.0     0.0   
1              0.0     1.0       1.0        1.0 -0.553424     1.0     1.0   
2              0.0     0.0       0.0        1.0 -0.553424     0.0     0.0   
3              0.0     1.0       1.0        1.0 -0.273623     1.0     0.0   
4              0.0     0.0       0.0        1.0 -0.133722     1.0     0.0   

   HeartDiseaseorAttack  PhysActivity  Fruits  ...  AnyHealthcare  \
0                   0.0           1.0     0.0  ...            1.0   
1                   0.0           0.0     1.0  ...            1.0   
2                   0.0           1.0     1.0  ...            1.0   
3                   0.0           1.0     1.0  ...            1.0   
4                   0.0           1.0     1.0  ...            1.0   

   NoDocbcCost   GenHlth  MentHlth  PhysHlth  DiffWalk  Sex       Age  \
0          0.0 -0.123028  0.140916  2.372482     

In [5]:
target = "Diabetes_binary"
X = df.drop(columns=[target])
y = df[target].astype(int)

# Split (estratificado recomendado en clasificación)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Configura y ejecuta FLAML (solo estimadores sklearn para evitar deps extra)
automl = AutoML()
settings = {
    "time_budget": 600,
    "task": "classification",
    "metric": "ap",          # PR-AUC
    "eval_method": "cv",
    "n_splits": 5,
    "seed": 42,
    "log_file_name": "flaml_automl.log",
    # Estimadores de sklearn (sin xgboost/lgbm/catboost):
    "estimator_list": ["lrl1","lrl2","rf","extra_tree","kneighbor","svc","sgd","histgb"],
}
automl.fit(X_train=X_train, y_train=y_train, **settings)

# Evaluación en test (métricas alineadas a tu benchmark)
y_pred  = automl.predict(X_test)
y_score = automl.predict_proba(X_test)[:, 1]

print("=== Métricas en TEST ===")
print("PR-AUC   :", average_precision_score(y_test, y_score))
print("F1       :", f1_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("ROC-AUC  :", roc_auc_score(y_test, y_score))
print("Accuracy :", accuracy_score(y_test, y_pred))

print("\n=== Modelo Seleccionado por FLAML ===")
print("Algoritmo:", automl.best_estimator)
print("Mejores hiperparámetros:\n", automl.best_config)

[flaml.automl.logger: 08-30 15:34:48] {1752} INFO - task = classification
[flaml.automl.logger: 08-30 15:34:48] {1763} INFO - Evaluation method: cv
[flaml.automl.logger: 08-30 15:34:48] {1862} INFO - Minimizing error metric: 1-ap
[flaml.automl.logger: 08-30 15:34:48] {1979} INFO - List of ML learners in AutoML Run: ['lrl1', 'lrl2', 'rf', 'extra_tree', 'kneighbor', 'svc', 'sgd', 'histgb']
[flaml.automl.logger: 08-30 15:34:48] {2282} INFO - iteration 0, current learner lrl1


INFO:flaml.tune.searcher.blendsearch:No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune


[flaml.automl.logger: 08-30 15:34:50] {2417} INFO - Estimated sufficient time budget=20498s. Estimated necessary time budget=20s.
[flaml.automl.logger: 08-30 15:34:50] {2466} INFO -  at 2.2s,	estimator lrl1's best error=0.2040,	best estimator lrl1's best error=0.2040
[flaml.automl.logger: 08-30 15:34:50] {2282} INFO - iteration 1, current learner svc


INFO:flaml.tune.searcher.blendsearch:No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune


[flaml.automl.logger: 08-30 15:34:51] {2466} INFO -  at 3.3s,	estimator svc's best error=0.2038,	best estimator svc's best error=0.2038
[flaml.automl.logger: 08-30 15:34:51] {2282} INFO - iteration 2, current learner sgd


INFO:flaml.tune.searcher.blendsearch:No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune


[flaml.automl.logger: 08-30 15:34:57] {2466} INFO -  at 9.6s,	estimator sgd's best error=0.2160,	best estimator svc's best error=0.2038
[flaml.automl.logger: 08-30 15:34:57] {2282} INFO - iteration 3, current learner histgb
[flaml.automl.logger: 08-30 15:34:57] {2466} INFO -  at 10.1s,	estimator histgb's best error=0.2985,	best estimator svc's best error=0.2038
[flaml.automl.logger: 08-30 15:34:57] {2282} INFO - iteration 4, current learner extra_tree
[flaml.automl.logger: 08-30 15:34:58] {2466} INFO -  at 10.4s,	estimator extra_tree's best error=0.2725,	best estimator svc's best error=0.2038
[flaml.automl.logger: 08-30 15:34:58] {2282} INFO - iteration 5, current learner rf
[flaml.automl.logger: 08-30 15:34:58] {2466} INFO -  at 10.7s,	estimator rf's best error=0.2406,	best estimator svc's best error=0.2038
[flaml.automl.logger: 08-30 15:34:58] {2282} INFO - iteration 6, current learner extra_tree
[flaml.automl.logger: 08-30 15:34:58] {2466} INFO -  at 11.1s,	estimator extra_tree's be

INFO:flaml.tune.searcher.blendsearch:No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune


[flaml.automl.logger: 08-30 15:35:08] {2466} INFO -  at 20.4s,	estimator lrl2's best error=0.2040,	best estimator svc's best error=0.2038
[flaml.automl.logger: 08-30 15:35:08] {2282} INFO - iteration 11, current learner rf
[flaml.automl.logger: 08-30 15:35:08] {2466} INFO -  at 20.8s,	estimator rf's best error=0.2292,	best estimator svc's best error=0.2038
[flaml.automl.logger: 08-30 15:35:08] {2282} INFO - iteration 12, current learner kneighbor
[flaml.automl.logger: 08-30 15:35:12] {2466} INFO -  at 24.6s,	estimator kneighbor's best error=0.2784,	best estimator svc's best error=0.2038
[flaml.automl.logger: 08-30 15:35:12] {2282} INFO - iteration 13, current learner rf
[flaml.automl.logger: 08-30 15:35:12] {2466} INFO -  at 25.0s,	estimator rf's best error=0.2197,	best estimator svc's best error=0.2038
[flaml.automl.logger: 08-30 15:35:12] {2282} INFO - iteration 14, current learner rf
[flaml.automl.logger: 08-30 15:35:13] {2466} INFO -  at 25.4s,	estimator rf's best error=0.2197,	bes